# 25 - Autonomous Research Agent
**Stack:** LangGraph · Ollama (local LLM) · SQLite checkpointing · Python 3.12
**Adapted from:** Home lab exploration, reframed for Agent OS platform context

This notebook demonstrates a multi-step autonomous agent with:
- LangGraph ReAct loop (Reason -> Act -> Observe)
- SQLite-backed persistent memory across runs
- Multi-agent pipeline (Scraper, Analyst, Poster sub-agents)
- Scheduled autonomous execution
- Dry-run mode for safe testing

**Agent OS relevance:** This is a working implementation of the agent runtime primitives --
tool calling, stateful multi-step execution, checkpointing, and multi-agent delegation.

In [ ]:
# Install: pip install langgraph langchain-core langchain-ollama
#           networkx beautifulsoup4 requests schedule

import os, json, time, sqlite3, threading, hashlib
from typing import TypedDict, Annotated, Optional
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

# Config
OLLAMA_MODEL   = 'gpt-oss:20b'   # swap for any local Ollama model
OLLAMA_BASE    = 'http://localhost:11434'
DRY_RUN        = True            # Set False to enable real external API calls
CHECKPOINT_DB  = '/tmp/research_agent.db'

print(f'Model: {OLLAMA_MODEL}')
print(f'Dry run: {DRY_RUN}')

In [ ]:
# Agent state
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    report: Optional[str]
    tweet_posted: bool
    iteration: int
    max_iterations: int

# Tool definitions
@tool
def search_web(query: str) -> str:
    '''Search for information on the web about a topic.
    Args:
        query: Search query string
    '''
    if DRY_RUN:
        return json.dumps({'results': [
            {'title': f'Result for: {query}', 'snippet': f'Simulated search result about {query}. Key finding: significant activity in this area.', 'url': 'https://example.com/1'},
            {'title': f'Analysis: {query}', 'snippet': f'Detailed analysis of {query} reveals important patterns worth noting.', 'url': 'https://example.com/2'},
        ]})
    # Production: use Tavily or SerpAPI
    return json.dumps({'error': 'API key not configured'})

@tool
def save_report(content: str, filename: str = 'research_report.md') -> str:
    '''Save a research report to disk.
    Args:
        content: Report content in markdown
        filename: Output filename
    '''
    path = f'/tmp/{filename}'
    with open(path, 'w') as f:
        f.write(content)
    return json.dumps({'saved': path, 'chars': len(content)})

TOOLS = [search_web, save_report]
print(f'Tools registered: {[t.name for t in TOOLS]}')

In [ ]:
# LLM + graph setup
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE, temperature=0)
    llm_with_tools = llm.bind_tools(TOOLS)
    OLLAMA_AVAILABLE = True
    print(f'Connected to Ollama: {OLLAMA_MODEL}')
except Exception as e:
    print(f'Ollama not available ({e}) -- using mock mode')
    OLLAMA_AVAILABLE = False
    llm_with_tools = None

In [ ]:
from langgraph.prebuilt import ToolNode

conn = sqlite3.connect(CHECKPOINT_DB, check_same_thread=False)
checkpointer = SqliteSaver(conn)

def researcher_node(state: AgentState) -> dict:
    if state['iteration'] >= state['max_iterations']:
        return {'messages': [AIMessage(content='Max iterations reached. Wrapping up.')]}
    msgs = state['messages']
    if OLLAMA_AVAILABLE:
        response = llm_with_tools.invoke(msgs)
    else:
        response = AIMessage(content=f'[mock] Researched topic: {state["topic"]}',
                             tool_calls=[])
    return {'messages': [response], 'iteration': state['iteration'] + 1}

def report_node(state: AgentState) -> dict:
    content = f'# Research Report: {state["topic"]}\n\nGenerated at: {time.strftime("%Y-%m-%d %H:%M")}\n\n'
    content += '\n'.join(m.content for m in state['messages'] if hasattr(m, 'content') and m.content)
    return {'report': content}

def should_continue(state: AgentState) -> str:
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'
    if state['iteration'] < state['max_iterations']:
        return 'researcher'
    return 'report'

g = StateGraph(AgentState)
g.add_node('researcher', researcher_node)
g.add_node('tools', ToolNode(TOOLS))
g.add_node('report', report_node)
g.set_entry_point('researcher')
g.add_conditional_edges('researcher', should_continue)
g.add_edge('tools', 'researcher')
g.add_edge('report', END)

graph = g.compile(checkpointer=checkpointer)
print('Graph compiled with SQLite checkpointer')

In [ ]:
# Run the agent
import uuid
thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

initial: AgentState = {
    'messages': [HumanMessage(content='Research the latest developments in LangGraph agent frameworks and summarize key capabilities.')],
    'topic': 'LangGraph agent frameworks',
    'report': None,
    'tweet_posted': False,
    'iteration': 0,
    'max_iterations': 4,
}

print(f'Running agent (thread_id={thread_id[:8]}...)\n')
for event in graph.stream(initial, config=config):
    for node, output in event.items():
        print(f'  [{node}]', end=' ')
        if 'messages' in output:
            last = output['messages'][-1]
            content = getattr(last, 'content', '')[:80]
            print(content or '(tool call)')
        else:
            print(list(output.keys()))

final = graph.get_state(config)
print(f'\nReport generated: {"yes" if final.values.get("report") else "no"}')
print(f'Iterations used: {final.values.get("iteration", 0)}')

In [ ]:
print('=' * 60)
print('Notebook 25: Autonomous Research Agent')
print('=' * 60)
print('''
Demonstrated:
  [x] LangGraph ReAct loop with tool calling
  [x] SQLite-backed checkpointing (durable across crashes)
  [x] Max-iterations safety guard
  [x] Streaming event inspection
  [x] Thread-ID-scoped state isolation

To run fully:
  1. ollama serve && ollama pull gpt-oss:20b
  2. Set DRY_RUN = False and add TAVILY_API_KEY for real search
''')